In [1]:
import pandas as pd
import numpy as np

In [2]:
manual = pd.read_csv('sarcasm_classifications_labeled.csv')
manual = manual[['headline', 'labels']]
manual

,headline,labels
0,side effects sound awesome,0
1,nation ready for its din din,1
2,pope francis wearing sweater vestments he got ...,1
3,nbc unveils on screen graphic informing audien...,1
4,"child baffled by stationary, non-violent images",0
...,...,...
595,"stan lee, creator of beloved marvel character ...",0
596,pfizer mercifully puts down another batch of t...,0
597,cern researchers apologize for destruction of ...,0
598,"'just take it slow, and you'll be fine,' drunk...",0


In [3]:
final = pd.read_csv('results_final.csv')
final

,headline,labels
0,historical archives: immoral woodcut discovere...,0
1,nation's tourists announce plans to form circl...,1
2,catherine zeta-jones happy to see people on in...,2
3,chick corea falls to communists,2
4,facebook clarifies site not intended to be use...,2
...,...,...
13628,ridley scott trades russell crowe to tim burto...,2
13629,son of edward r. murrow says father 'real dirt...,2
13630,loser older brother looked up to,1
13631,new social media start-up aims to be cross bet...,0


In [4]:
# Check for exact duplicates within final itself
duplicate_rows = final[final.duplicated(['headline'], keep=False)]
print(f"\nNumber of duplicate headlines within final dataset: {len(duplicate_rows)}")

# Drop duplicates from final keeping the first occurrence
final_deduplicated = final.drop_duplicates(['headline'], keep='first')
print(f"\nOriginal final shape: {final.shape}")
print(f"Deduplicated final shape: {final_deduplicated.shape}")
print(f"Removed {final.shape[0] - final_deduplicated.shape[0]} duplicate rows")

final_deduplicated


Number of duplicate headlines within final dataset: 150

Original final shape: (13633, 2)
Deduplicated final shape: (13551, 2)
Removed 82 duplicate rows


,headline,labels
0,historical archives: immoral woodcut discovere...,0
1,nation's tourists announce plans to form circl...,1
2,catherine zeta-jones happy to see people on in...,2
3,chick corea falls to communists,2
4,facebook clarifies site not intended to be use...,2
...,...,...
13628,ridley scott trades russell crowe to tim burto...,2
13629,son of edward r. murrow says father 'real dirt...,2
13630,loser older brother looked up to,1
13631,new social media start-up aims to be cross bet...,0


In [5]:
final = final_deduplicated

In [6]:
# Merge with indicator to identify rows that are only in 'final'
merged = pd.merge(final, manual, on=['headline', 'labels'], how='left', indicator=True)

# Keep only rows that are in 'final' but not in 'manual'
final_filtered = merged[merged['_merge'] == 'left_only'].drop('_merge', axis=1)

print(f"Original final shape: {final.shape}")
print(f"Filtered final shape: {final_filtered.shape}")
final_filtered

Original final shape: (13551, 2)
Filtered final shape: (12951, 2)


,headline,labels
0,historical archives: immoral woodcut discovere...,0
1,nation's tourists announce plans to form circl...,1
2,catherine zeta-jones happy to see people on in...,2
3,chick corea falls to communists,2
4,facebook clarifies site not intended to be use...,2
...,...,...
13545,white house press corps wishes show of solidar...,2
13546,ridley scott trades russell crowe to tim burto...,2
13547,son of edward r. murrow says father 'real dirt...,2
13548,loser older brother looked up to,1


In [7]:
from sklearn.model_selection import train_test_split

print("Original label distribution:")
print(final_filtered['labels'].value_counts())

stratified_sample = pd.DataFrame()
for label in [0, 1, 2]:
    label_data = final_filtered[final_filtered['labels'] == label]
    if len(label_data) >= 100:
        sample, _ = train_test_split(label_data, 
                                    train_size=100, 
                                    random_state=42,
                                    stratify=None)  # No need to stratify within a single label
        stratified_sample = pd.concat([stratified_sample, sample], ignore_index=True)
    else:
        print(f"Warning: Label {label} has fewer than 100 samples. Taking all {len(label_data)} samples.")
        stratified_sample = pd.concat([stratified_sample, label_data], ignore_index=True)

# Shuffle the final dataset
stratified_sample = stratified_sample.sample(frac=1, random_state=42).reset_index(drop=True)

# Check the distribution in the sampled dataset
print("\nSampled label distribution:")
print(stratified_sample['labels'].value_counts())

# Display sample size and first few rows
print(f"\nTotal sample size: {len(stratified_sample)}")
stratified_sample

Original label distribution:
2    4369
1    4296
0    4286
Name: labels, dtype: int64

Sampled label distribution:
2    100
1    100
0    100
Name: labels, dtype: int64

Total sample size: 300


,headline,labels
0,pence unveils campaign to educate teens about ...,2
1,new york to host 1998 ill-will games,2
2,double-jointed man on date breaks it out too e...,1
3,smoker inspired by sight of elderly smoker,0
4,corner store customers saddened by sight of fr...,2
...,...,...
295,"not snowing over here, man on phone reports",1
296,jim morrison stares creepily out of apartment ...,0
297,youngest sibling in family kind of thought mom...,1
298,nation shudders to think how bad things would ...,2


In [8]:
stratified_sample.to_csv('stratified_sampled_w_label.csv', index=False)

In [9]:
stratified_sample_no_label = stratified_sample[['headline']]
stratified_sample_no_label.to_csv('stratified_sampled_wo_label.csv', index=False)

In [10]:
rest_of_dataset = final_filtered[~final_filtered['headline'].isin(stratified_sample['headline'])]

print(f"Original dataset size: {len(final_filtered)}")
print(f"Sampled dataset size: {len(stratified_sample)}")
print(f"Rest of dataset size: {len(rest_of_dataset)}")


print("\nLabel distribution in the rest of the dataset:")
print(rest_of_dataset['labels'].value_counts())

rest_of_dataset

Original dataset size: 12951
Sampled dataset size: 300
Rest of dataset size: 12651

Label distribution in the rest of the dataset:
2    4269
1    4196
0    4186
Name: labels, dtype: int64


,headline,labels
0,historical archives: immoral woodcut discovere...,0
1,nation's tourists announce plans to form circl...,1
2,catherine zeta-jones happy to see people on in...,2
3,chick corea falls to communists,2
4,facebook clarifies site not intended to be use...,2
...,...,...
13545,white house press corps wishes show of solidar...,2
13546,ridley scott trades russell crowe to tim burto...,2
13547,son of edward r. murrow says father 'real dirt...,2
13548,loser older brother looked up to,1


In [11]:
rest_of_dataset.to_csv('train.csv', index=False)

In [ ]:
test = pd.concat([stratified_sample_no_label, manual[['headline']]])
test

,headline
0,pence unveils campaign to educate teens about ...
1,new york to host 1998 ill-will games
2,double-jointed man on date breaks it out too e...
3,smoker inspired by sight of elderly smoker
4,corner store customers saddened by sight of fr...
...,...
595,"stan lee, creator of beloved marvel character ..."
596,pfizer mercifully puts down another batch of t...
597,cern researchers apologize for destruction of ...
598,"'just take it slow, and you'll be fine,' drunk..."


In [13]:
test.to_csv('test.csv', index=False)